<a href="https://colab.research.google.com/github/evieweird/Team1_AI/blob/main/Team1_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------------------------------------------
# ------------------------Required Libraries ------------------------
# -------------------------------------------------------------------
!pip -q install transformers albumentations
!pip -q install segmentation-models-pytorch
!pip -q install segmentation-models-pytorch albumentations
!pip -q install kaggle

In [ ]:
# -------------------------------------------------------------------
# ------------------------Dataset Preparation------------------------
# -------------------------------------------------------------------

# Import libraries
from pathlib import Path
import os, zipfile
import pandas as pd
from PIL import Image, ImageDraw
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# Define data directory and create it if it doesn't exist
DATA_DIR = "/content/ddti_data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download the dataset from Kaggle
!kaggle datasets download -d dasmehdixtr/ddti-thyroid-ultrasound-images -p $DATA_DIR

# Unzip the downloaded dataset files
for f in os.listdir(DATA_DIR):
    if f.endswith(".zip"):
        with zipfile.ZipFile(os.path.join(DATA_DIR, f), "r") as z:
            z.extractall(DATA_DIR)

print("Dataset ready")

# Get a list of all image files (JPG) in the data directory
image_files = set([f for f in os.listdir(DATA_DIR) if f.lower().endswith(".jpg")])

rows = []

# List to store unique image dimensions (all images are expected to be the same size)
unique_dims = []

# Get a sorted list of all XML annotation files
xml_files = sorted([f for f in os.listdir(DATA_DIR) if f.lower().endswith(".xml")])

# Process each XML file to extract annotations and pair with images
for xml_name in xml_files:
    xml_path = os.path.join(DATA_DIR, xml_name)

    try:
        # Parse the XML file
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Extract case number
        case_number = root.find("number")
        if case_number is None or case_number.text is None:
            continue

        case_number = case_number.text.strip()

        # Iterate through 'mark' elements to find image and SVG data
        for mark in root.findall("mark"):
            image_elem = mark.find("image")
            svg_elem = mark.find("svg")

            if image_elem is None or svg_elem is None:
                continue
            if image_elem.text is None or svg_elem.text is None:
                continue

            image_idx = image_elem.text.strip()
            svg_text = svg_elem.text.strip()

            # Construct image name and check if it exists
            image_name = f"{case_number}_{image_idx}.jpg"

            if image_name in image_files:
                # Add data to the rows list
                rows.append({
                    "case_id": case_number,
                    "image_path": os.path.join(DATA_DIR, image_name),
                    "xml_path": xml_path,
                    "svg": svg_text,
                    "mask_path": ""
                })
                # Get image dimensions and add to unique_dims if new
                dims = Image.open(os.path.join(DATA_DIR, image_name)).size
                if dims not in unique_dims:
                    unique_dims.append(dims)

    except Exception as e:
        print(f"Skipping {xml_name}: {e}")

# Create a DataFrame from the extracted data
df = pd.DataFrame(rows)
case_ids = df["case_id"].unique()
print(unique_dims) #all images are 560x360

# 1. Convert SVG polygon annotations into binary segmentation masks

# Assume all images have the same dimensions
image_size = unique_dims[0]

# Create a mask for each image based on the SVG provided
no_mask = 0  # Counter for SVGs that are empty
with_syntax_error = 0  # Counter for XMLs with syntax errors

# Create a directory for saving masks
output_folder = Path(os.path.join(DATA_DIR, "masks"))
output_folder.mkdir(parents=True, exist_ok=True)

# Iterate through the DataFrame to generate masks
for idx, row in df.iterrows():
    # Create a blank single-channel (binary) image for the mask
    img = Image.new("1", image_size)
    draw = ImageDraw.Draw(img)
    svg_str = row['svg']
    case_num = row['case_id']
    image_path = row['image_path']

    if svg_str is None:
        no_mask += 1
        print(f"No mask for {image_path}")
        continue
    try:
        # Convert SVG string to Python object (list of dictionaries)
        svg_content = eval(svg_str)
    except SyntaxError:
        with_syntax_error += 1
        print(f"Syntax error for {image_path}")
        continue

    # Only proceed if there is actual polygon data in the SVG content
    if svg_content and len(svg_content) > 0:
        for area in svg_content:
            # Extract polygon points and draw on the mask image
            points = [(point["x"], point["y"]) for point in area["points"]]
            draw.polygon(points, fill='white') # Fill polygon with white for mask
        # Define mask path and save the generated mask
        mask_path = os.path.join(output_folder, case_num + "_mask.png")
        # Save the mask path back into the DataFrame
        df.at[idx, 'mask_path'] = mask_path
        img.save(mask_path)

print(f"Total images: {len(df)}")
# Filter out rows where mask generation failed (mask_path is empty)
df = df[df['mask_path'] != ""].reset_index(drop=True)

print(f"Total usable images with masks: {len(df)}")

# Split data into training, validation, and test sets based on case_id
# This ensures that images from the same patient (case_id) are not split across sets
train_cases, temp_cases = train_test_split(case_ids, test_size=0.2, random_state=42)
val_cases, test_cases = train_test_split(temp_cases, test_size=0.5, random_state=42)

# Create separate DataFrames for train, validation, and test sets
train_df = df[df["case_id"].isin(train_cases)].reset_index(drop=True)
val_df = df[df["case_id"].isin(val_cases)].reset_index(drop=True)
test_df = df[df["case_id"].isin(test_cases)].reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# -------------------------------------------------------------------
# ----Load ultrasound images/masks. Augmentation and verify checks---
# -------------------------------------------------------------------
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from PIL import Image, ImageDraw
import ast
from pathlib import Path
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Dataset class for loading ultrasound images and their corresponding masks
class UltrasoundDataset(Dataset):
    def __init__(self, rows, image_size=(560, 360), transform=None):
        self.rows = rows # DataFrame containing image and mask paths
        self.image_size = image_size # Expected size of the images
        self.transform = transform # Transformations to apply to images and masks

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]

        # Load image and mask
        image = Image.open(row['image_path']).convert("RGB") # Convert image to RGB
        mask = Image.open(row['mask_path']).convert("L") # Convert mask to grayscale

        # Convert PIL Images to NumPy arrays
        image = np.array(image)
        mask = np.array(mask)
        # Convert mask to binary (0 or 1) float32 array
        mask = (mask > 0).astype(np.float32)

        # Apply transformations if provided (e.g., augmentation, normalization)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long() # Return image and mask, mask as long for CrossEntropyLoss

# Define target height and width for resizing images
TARGET_H, TARGET_W = 384, 576

# Data augmentation pipeline for training data
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.Affine(scale=(0.85, 1.15), rotate=(-20, 20), translate_percent=(-0.1, 0.1), p=0.5),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
    A.GridDistortion(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(16, 32), hole_width_range=(16, 32), p=0.3),
    A.Resize(TARGET_H, TARGET_W), # Resize to target dimensions
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalize image pixel values
    ToTensorV2(), # Convert NumPy arrays to PyTorch tensors
])

# Data augmentation pipeline for validation and test data (only resizing and normalization)
val_aug = A.Compose([
    A.Resize(TARGET_H, TARGET_W),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Create Dataset Objects for training, validation, and testing
train_ds = UltrasoundDataset(train_df, transform=train_aug)
val_ds = UltrasoundDataset(val_df, transform=val_aug)
test_ds = UltrasoundDataset(test_df, transform=val_aug)

# Create DataLoader Objects for batching and shuffling data
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False)

# Quick verification check: print shapes of image and mask batches
images, masks = next(iter(train_loader))
print(f"Image batch shape: {images.shape}") # Expect ?[batch_size, 3, TARGET_H, TARGET_W]
print(f"Mask batch shape: {masks.shape}")   # Expect ?[batch_size, TARGET_H, TARGET_W]

def verify_paths(df):
    """
    Verifies if the image and mask paths in a DataFrame exist on the file system.
    Prints a message indicating if all files are found or lists missing files.

    Args:
        df (pd.DataFrame): DataFrame containing 'image_path' and 'mask_path' columns.
    """
    missing_files = []
    for i, row in df.iterrows():
        # Check if the image file exists
        if not os.path.exists(row['image_path']):
            missing_files.append(f"MISSING IMAGE: {row['image_path']} at index {i}")
        # Check if the mask file exists
        if not os.path.exists(row['mask_path']):
            missing_files.append(f"MISSING MASK: {row['mask_path']} at index {i}")

    if missing_files:
        print(f"Found {len(missing_files)} missing files:")
        for msg in missing_files[:5]: # Show only the first 5 missing files for brevity
            print(msg)
    else:
        print("All file paths verified!")

# Run the verification on the training DataFrame
verify_paths(train_df)


Image batch shape: torch.Size([4, 3, 384, 576])
Mask batch shape: torch.Size([4, 384, 576])


In [ ]:
# -------------------------------------------------------------------
# ------------Model: UNet++ with EfficientNet-B4 encoder-------------
# -------------------------------------------------------------------
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm import tqdm
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Determine if CUDA (GPU) is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Clear CUDA cache to free up GPU memory
torch.cuda.empty_cache()
import gc; gc.collect()

# Model Definition: UNet++ with EfficientNet-B4 encoder
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b4", # Use EfficientNet-B4 as the backbone encoder
    encoder_weights="imagenet",    # Initialize encoder with weights pre-trained on ImageNet
    in_channels=3,                 # Input images have 3 channels (RGB)
    classes=1,                     # Output is a single-channel binary mask
    activation=None,               # No activation function at the output layer (BCEWithLogitsLoss handles it)
    decoder_dropout=0.25,          # Apply dropout to the decoder to prevent overfitting
)
model = model.to(device) # Move the model to the selected device (GPU or CPU)

# Loss Function: Balanced Dice + BCE Loss
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss() # Binary Cross-Entropy with Logits Loss
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        # Ensure target has an extra channel dimension for BCE loss calculation
        target = target.float().unsqueeze(1)

        # Calculate Binary Cross-Entropy Loss
        bce_loss = self.bce(pred, target)

        # Calculate Dice Loss
        pred_sig = torch.sigmoid(pred) # Apply sigmoid to predictions to get probabilities
        smooth = 1e-5 # Smoothing factor to prevent division by zero
        intersection = (pred_sig * target).sum(dim=(2, 3)) # Sum over height and width
        union = pred_sig.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice_score = (2.0 * intersection + smooth) / (union + smooth)
        dice_loss = 1.0 - dice_score.mean() # Dice Loss is 1 - Dice Score

        # Combine BCE and Dice losses with specified weights
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# Instantiate the custom loss function
criterion = DiceBCELoss(bce_weight=0.5, dice_weight=0.5)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

NUM_EPOCHS = 15 # Total number of training epochs

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6 # Cosine annealing schedule with warm restarts
)

# Metrics
THRESHOLDS = np.arange(0.30, 0.70, 0.05) # Thresholds to evaluate for Dice and IoU scores

def compute_metrics(pred, target, threshold):
    """Computes IoU and Dice scores for a given prediction, target, and threshold."""
    pred = torch.sigmoid(pred) # Convert logits to probabilities
    pred = (pred > threshold).float().squeeze(1) # Binarize predictions and remove channel dim
    target = target.float() # Ensure target is float for calculations

    smooth = 1e-5 # Smoothing factor
    intersection = (pred * target).sum(dim=(1, 2)) # Intersection area
    union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) - intersection # Union area

    iou = ((intersection + smooth) / (union + smooth)).mean().item() # Intersection over Union
    dice = ((2 * intersection + smooth) /
            (pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) + smooth)).mean().item() # Dice coefficient
    return iou, dice


def best_threshold_metrics(preds, masks):
    """Return best (iou, dice) across thresholds."""
    best_iou, best_dice = 0.0, 0.0
    for t in THRESHOLDS:
        iou, dice = compute_metrics(preds, masks, threshold=t)
        if dice > best_dice:
            best_iou, best_dice = iou, dice
    return best_iou, best_dice


# Training loop initialization
best_val_dice = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 15 # Number of epochs to wait for improvement before early stopping

# Lists to store training and validation metrics for plotting
train_losses, val_losses = [], []
train_dices, val_dices = [], []
train_ious, val_ious = [], []

# Main training loop
for epoch in range(NUM_EPOCHS):
    # --- Training Phase ---
    model.train() # Set model to training mode
    epoch_loss = 0.0
    epoch_iou, epoch_dice = 0.0, 0.0

    # Iterate over training data batches
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images = images.to(device) # Move images to the device
        masks = masks.to(device)   # Move masks to the device

        optimizer.zero_grad() # Clear gradients from the previous step
        preds = model(images) # Forward pass: get predictions
        loss = criterion(preds, masks) # Compute loss
        loss.backward() # Backward pass: compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Clip gradients to prevent exploding gradients
        optimizer.step() # Update model weights

        epoch_loss += loss.item() # Accumulate batch loss
        iou, dice = best_threshold_metrics(preds.detach(), masks) # Compute metrics (detach predictions to avoid computing gradients for metrics)
        epoch_iou += iou
        epoch_dice += dice

    scheduler.step(epoch) # Update learning rate scheduler

    n_train = len(train_loader)
    train_losses.append(epoch_loss / n_train) # Average training loss for the epoch
    train_ious.append(epoch_iou / n_train)
    train_dices.append(epoch_dice / n_train)

    # --- Validation Phase ---
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    v_iou, v_dice = 0.0, 0.0

    # Iterate over validation data batches without gradient computation
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images = images.to(device)
            masks = masks.to(device)

            preds = model(images)
            loss = criterion(preds, masks)

            val_loss += loss.item()
            iou, dice = best_threshold_metrics(preds, masks)
            v_iou += iou
            v_dice += dice

    n_val = len(val_loader)
    val_losses.append(val_loss / n_val) # Average validation loss for the epoch
    val_ious.append(v_iou / n_val)
    val_dices.append(v_dice / n_val)

    # Save best model + early stopping logic
    if val_dices[-1] > best_val_dice:
        best_val_dice = val_dices[-1]
        torch.save(model.state_dict(), "best_cnn_model.pth") # Save model if validation Dice improves
        patience_counter = 0 # Reset patience counter
    else:
        patience_counter += 1 # Increment patience counter if no improvement

    current_lr = optimizer.param_groups[0]['lr'] # Get current learning rate
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  (lr={current_lr:.2e})")
    print(f"  Train Loss: {train_losses[-1]:.4f} | Dice: {train_dices[-1]:.4f} | IoU: {train_ious[-1]:.4f}")
    print(f"  Val   Loss: {val_losses[-1]:.4f} | Dice: {val_dices[-1]:.4f} | IoU: {val_ious[-1]:.4f}")
    print(f"  Best Val Dice: {best_val_dice:.4f}")
    print("-" * 50)

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break # Stop training if early stopping patience is exceeded

# Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_losses, label="Train")
axes[0].plot(val_losses, label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(train_dices, label="Train")
axes[1].plot(val_dices, label="Val")
axes[1].set_title("Dice Score")
axes[1].legend()

axes[2].plot(train_ious, label="Train")
axes[2].plot(val_ious, label="Val")
axes[2].set_title("IoU")
axes[2].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

# Test Evaluation with test-time augmentation (TTA)
model.load_state_dict(torch.load("best_cnn_model.pth")) # Load the best trained model weights
model.eval() # Set model to evaluation mode


def tta_predict(model, images):
    """Average predictions over original + flipped versions."""
    # Original prediction
    preds = torch.sigmoid(model(images))

    # Horizontal flip augmentation
    flipped_h = torch.flip(images, dims=[3]) # Flip horizontally
    preds += torch.flip(torch.sigmoid(model(flipped_h)), dims=[3]) # Predict and flip back

    # Vertical flip augmentation
    flipped_v = torch.flip(images, dims=[2]) # Flip vertically
    preds += torch.flip(torch.sigmoid(model(flipped_v)), dims=[2]) # Predict and flip back

    return preds / 3.0 # Average the predictions from original and flipped images


# Find optimal threshold on validation set
print("Finding optimal threshold on validation set...")
best_test_thresh = 0.5
best_thresh_dice = 0.0

with torch.no_grad():
    for t in np.arange(0.25, 0.75, 0.05): # Iterate through a range of thresholds
        total_dice = 0.0
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            preds_raw = tta_predict(model, images) # Get raw predictions using TTA
            pred_bin = (preds_raw > t).float().squeeze(1) # Binarize predictions
            smooth = 1e-5
            inter = (pred_bin * masks.float()).sum(dim=(1, 2))
            dice = ((2 * inter + smooth) /
                    (pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) + smooth)).mean().item()
            total_dice += dice
        avg_dice = total_dice / len(val_loader)
        if avg_dice > best_thresh_dice:
            best_thresh_dice = avg_dice
            best_test_thresh = t # Store the threshold that yields the best Dice score

print(f"Optimal threshold: {best_test_thresh:.2f} (val dice: {best_thresh_dice:.4f})")

# Final test evaluation
test_iou, test_dice = 0.0, 0.0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        preds_raw = tta_predict(model, images)
        pred_bin = (preds_raw > best_test_thresh).float().squeeze(1) # Apply optimal threshold

        smooth = 1e-5
        inter = (pred_bin * masks.float()).sum(dim=(1, 2))
        union = pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) - inter

        test_iou += ((inter + smooth) / (union + smooth)).mean().item()
        test_dice += ((2 * inter + smooth) /
                      (pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) + smooth)).mean().item()

n_test = len(test_loader)
print(f"Test Dice: {test_dice / n_test:.4f}")
print(f"Test IoU:  {test_iou / n_test:.4f}")

# Visualize predictions
images, masks = next(iter(test_loader)) # Get a batch of test images and masks
images = images.to(device)

with torch.no_grad():
    preds = tta_predict(model, images) # Get predictions
    preds = (preds > best_test_thresh).float().cpu() # Binarize predictions and move to CPU

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("UNet++/EfficientNet-B4 Predictions on Thyroid Ultrasound Dataset", fontsize=14)

for i in range(3):
    img = images[i].cpu()
    # De-normalize image for display
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + \
          torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    img = img.permute(1, 2, 0).clamp(0, 1) # Permute dimensions for imshow and clamp values

    axes[i][0].imshow(img)
    axes[i][0].set_title("Input Thyroid Ultrasound")
    axes[i][1].imshow(masks[i], cmap="gray")
    axes[i][1].set_title("Ground Truth")
    axes[i][2].imshow(preds[i].squeeze(), cmap="gray")
    axes[i][2].set_title("Prediction")

for ax in axes.flat:
    ax.axis("off") # Turn off axes for cleaner visualization
plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.savefig("predictions.png", dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------
# ---------------Model: SegFormer w/ MiT-B2 backbone-----------------
# -------------------------------------------------------------------
best_val_dice = 0.0
patience_counter = 0.0

# Data already prepared

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

# Load SegFormer w/ MiT-B2 backbone
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vit_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=2, # background (0), nodule (1)
    ignore_mismatched_sizes=True  # wil llet us swap in new classification head
)

vit_model = vit_model.to(device)
print("\n----------------------------------------------")
print("\nSegFormer model loaded sucessfully")

# Helper Function - umpsamples predictions back to full mask size so we can compare to ground truth masks
# since SegFormer outputs logits at 1/4 input resolution

def segformer_predict(model, images):
  """
    Run SegFormer and return full-resolution logits.
    images: tensor of shape [B, 3, H, W]
    returns: tensor of shape [B, 2, H, W]
  """

  outputs = model(pixel_values=images)
  logits = outputs.logits
  logits = F.interpolate(
      logits,
      size = images.shape[-2:], #(H,W)
      mode = "bilinear",
      align_corners=False
  )
  return logits #shape: [b, 2, h, w]

# Loss Function & Metrics

# ----- Loss Function -----
# Combines two losses:
#   BCE (Binary Cross Entropy) penalizes wrong pixel classifications
#   Dice penalizes poor overlap between predicted mask and ground truth
# Using both together gives more stable training than either alone
class DiceCELoss(nn.Module):
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()  # For 2-class output
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, target):
        # logits: [B, 2, H, W], target: [B, H, W] with values 0 or 1
        ce_loss = self.ce(logits, target)

        # For Dice, get probability of class 1 (nodule)
        probs = torch.softmax(logits, dim=1)[:, 1, :, :]  # [B, H, W]
        target_f = target.float()
        smooth = 1e-5
        intersection = (probs * target_f).sum(dim=(1, 2))
        union = probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
        dice_score = (2.0 * intersection + smooth) / (union + smooth)
        dice_loss = 1.0 - dice_score.mean()

        return self.ce_weight * ce_loss + self.dice_weight * dice_loss

criterion = DiceCELoss(ce_weight=0.5, dice_weight=0.5)

# ----- Metrics -----
# Dice Score measures overlap (0 = no overlap, 1 = perfect)
THRESHOLDS = np.arange(0.30, 0.70, 0.05)

def compute_metrics(logits, target):
    """
    Compute best Dice and IoU across multiple thresholds.
    logits: [B, 2, H, W]
    target: [B, H, W]
    """
    probs = torch.softmax(logits, dim=1)[:, 1, :, :]  # probability of being a nodule
    target_f = target.float()
    best_iou, best_dice = 0.0, 0.0
    smooth = 1e-5
    for t in THRESHOLDS:
        pred = (probs > t).float()
        intersection = (pred * target_f).sum(dim=(1, 2))
        union = pred.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) - intersection
        iou  = ((intersection + smooth) / (union + smooth)).mean().item()
        dice = ((2 * intersection + smooth) /
                (pred.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) + smooth)).mean().item()
        if dice > best_dice:
            best_iou, best_dice = iou, dice
    return best_iou, best_dice

print("Loss function and metrics ready")

# Training the ViT Model
from tqdm import tqdm

# optimizer adjusts the model weights each step to reduce loss
optimizer = torch.optim.AdamW(vit_model.parameters(), lr=6e-5, weight_decay=1e-4)

NUM_EPOCHS = 15

# scheduler to help fine tune
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Track losses and metrics for plotting later
vit_train_losses, val_losses = [], []
vit_train_dices, val_dices = [], []
vit_train_ious, val_ious = [], []

best_val_dice = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 15  # Stop if no improvement for 15 epochs

for epoch in range(NUM_EPOCHS):

    # Training phase:
    vit_model.train()
    epoch_loss = 0.0
    epoch_iou, epoch_dice = 0.0, 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()          # Clear gradients from last step
        logits = segformer_predict(vit_model, images)  # Forward pass
        loss = criterion(logits, masks)                # Compute loss
        loss.backward()                                # Backpropagation
        torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)  # Prevent exploding gradients
        optimizer.step()               # Update weights

        epoch_loss += loss.item()
        iou, dice = compute_metrics(logits.detach(), masks)
        epoch_iou  += iou
        epoch_dice += dice

    scheduler.step(epoch)
    n_train = len(train_loader)
    vit_train_losses.append(epoch_loss / n_train)
    vit_train_ious.append(epoch_iou   / n_train)
    vit_train_dices.append(epoch_dice / n_train)

    # Validation Phase:
    # doesn't update weights, just measures
    vit_model.eval()
    val_loss = 0.0
    v_iou, v_dice = 0.0, 0.0

    with torch.no_grad():  # no gradient tracking needed during eval
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images = images.to(device)
            masks  = masks.to(device)
            logits = segformer_predict(vit_model, images)
            loss   = criterion(logits, masks)
            val_loss += loss.item()
            iou, dice = compute_metrics(logits, masks)
            v_iou  += iou
            v_dice += dice

    n_val = len(val_loader)
    val_losses.append(val_loss / n_val)
    val_ious.append(v_iou   / n_val)
    val_dices.append(v_dice / n_val)

    # Saves the model if it's the best we've seen so far
    if val_dices[-1] > best_val_dice:
        best_val_dice = val_dices[-1]
        torch.save(vit_model.state_dict(), "best_vit_model.pth")
        patience_counter = 0
        print("New best model saved")
    else:
        patience_counter += 1

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  (lr={current_lr:.2e})")
    print(f"  Train Loss: {vit_train_losses[-1]:.4f} | Dice: {vit_train_dices[-1]:.4f} | IoU: {vit_train_ious[-1]:.4f}")
    print(f"  Val   Loss: {val_losses[-1]:.4f} | Dice: {val_dices[-1]:.4f} | IoU: {val_ious[-1]:.4f}")
    print(f"  Best Val Dice: {best_val_dice:.4f}")
    print("-" * 50)

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("Training complete!")

# Plot ViT Training Curves

import matplotlib.pyplot as plt # Ensure matplotlib is imported for plotting

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("SegFormer (ViT) Training Curves", fontsize=14)

# Plot Training and Validation Loss
axes[0].plot(vit_train_losses, label="Train")
axes[0].plot(val_losses,   label="Val")
axes[0].set_title("Loss")
axes[0].legend()

# Plot Training and Validation Dice Score
axes[1].plot(vit_train_dices, label="Train")
axes[1].plot(val_dices,   label="Val")
axes[1].set_title("Dice Score")
axes[1].legend()

# Plot Training and Validation IoU
axes[2].plot(vit_train_ious, label="Train")
axes[2].plot(val_ious,   label="Val")
axes[2].set_title("IoU")
axes[2].legend()

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.savefig("vit_training_curves.png", dpi=150) # Save the plot as a PNG file
plt.show()


# Test evaluation
# Load the best model weights saved during training
vit_model.load_state_dict(torch.load("best_vit_model.pth"))
vit_model.eval()

# Test-Time Augmentation (TTA):
# We run each test image through the model 3 times:
#   1. Normal
#   2. Horizontally flipped, then flip prediction back
#   3. Vertically flipped, then flip prediction back
# Averaging these 3 predictions reduces noise and improves accuracy
def tta_predict_vit(model, images):
    probs = torch.softmax(segformer_predict(model, images), dim=1)[:, 1:, :, :]

    flipped_h = torch.flip(images, dims=[3])
    probs += torch.flip(torch.softmax(segformer_predict(model, flipped_h), dim=1)[:, 1:, :, :], dims=[3])

    flipped_v = torch.flip(images, dims=[2])
    probs += torch.flip(torch.softmax(segformer_predict(model, flipped_v), dim=1)[:, 1:, :, :], dims=[2])

    return probs / 3.0  # Average the 3 predictions

# Find the best threshold on validation set
print("Finding optimal threshold on validation set...")
best_thresh = 0.5
best_thresh_dice = 0.0

with torch.no_grad():
    for t in np.arange(0.25, 0.75, 0.05):
        total_dice = 0.0
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            probs = tta_predict_vit(vit_model, images).squeeze(1)
            pred_bin = (probs > t).float()
            smooth = 1e-5
            inter = (pred_bin * masks.float()).sum(dim=(1, 2))
            dice = ((2 * inter + smooth) /
                    (pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) + smooth)).mean().item()
            total_dice += dice
        avg_dice = total_dice / len(val_loader)
        if avg_dice > best_thresh_dice:
            best_thresh_dice = avg_dice
            best_thresh = t

print(f"Optimal threshold: {best_thresh:.2f}  (val Dice: {best_thresh_dice:.4f})")

#Final test evaluation:
vit_test_iou, vit_test_dice = 0.0, 0.0

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        probs    = tta_predict_vit(vit_model, images).squeeze(1)
        pred_bin = (probs > best_thresh).float()
        smooth   = 1e-5
        inter    = (pred_bin * masks.float()).sum(dim=(1, 2))
        union    = pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) - inter
        vit_test_iou  += ((inter + smooth) / (union + smooth)).mean().item()
        vit_test_dice += ((2 * inter + smooth) /
                          (pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) + smooth)).mean().item()

n_test = len(test_loader)
vit_test_dice = vit_test_dice / n_test
vit_test_iou  = vit_test_iou  / n_test

print(f"\n=== ViT (SegFormer) Test Results ===")
print(f"Test Dice: {vit_test_dice:.4f}")
print(f"Test IoU:  {vit_test_iou:.4f}")

# Visual Predictions
# Grab one batch from the test set
images, masks = next(iter(test_loader))
images_gpu = images.to(device)

with torch.no_grad():
    probs = tta_predict_vit(vit_model, images_gpu).squeeze(1).cpu()
    preds = (probs > best_thresh).float()

# Show 3 examples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("SegFormer (ViT) Predictions on Thyroid Ultrasound Dataset", fontsize=14)

for i in range(3):
    # De-normalize the image for display (undo the normalization we applied earlier)
    img = images[i].clone()
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + \
              torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    img = img.permute(1, 2, 0).clamp(0, 1)

    axes[i][0].imshow(img)
    axes[i][0].set_title("Input Thyroid Ultrasound")

    axes[i][1].imshow(masks[i], cmap="gray")
    axes[i][1].set_title("Ground Truth Mask")

    axes[i][2].imshow(preds[i].squeeze(), cmap="gray")
    axes[i][2].set_title("ViT Prediction")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.savefig("vit_predictions.png", dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------
# ------------------UNet++ vs SegFormer Comparison-------------------
# -------------------------------------------------------------------

# Assuming these variables (cnn_test_dice, cnn_test_iou) are populated from the UNet++ evaluation.
# If not, you might need to ensure they are defined in the previous cells or manually set.
cnn_test_dice = 0.77
cnn_test_iou  = 0.67

# Print comparison table to show the performance of both models
print("\n" + "="*45)
print(f"{"Model":<25} {"Dice":>8} {"IoU":>8}")
print("-"*45)
print(f"{"UNet++ / CNN":<25} {cnn_test_dice:>8.4f} {cnn_test_iou:>8.4f}")
print(f"{"SegFormer / ViT":<25} {vit_test_dice:>8.4f} {vit_test_iou:>8.4f}")
print("="*45)

# Bar chart comparison to visualize the performance metrics
models  = ["UNet++ (CNN)", "SegFormer (ViT)"]
dices   = [cnn_test_dice, vit_test_dice]
ious    = [cnn_test_iou,  vit_test_iou]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
# Plot Dice scores
bars1 = ax.bar(x - width/2, dices, width, label="Dice Score", color="steelblue")
# Plot IoU scores
bars2 = ax.bar(x + width/2, ious,  width, label="IoU",        color="coral")

# Add values on top of the bars for better readability
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=10)

ax.set_ylabel("Score")
ax.set_title("CNN vs ViT — Thyroid Ultrasound Segmentation")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.savefig("cnn_vs_vit_comparison.png", dpi=150)
plt.show()

In [ ]:
# -------------------------------------------------------------------
# ---------------------Download BUS-BRA Dataset----------------------
# -------------------------------------------------------------------
import os, zipfile, glob

# Define the directory for the breast ultrasound data
BREAST_DIR = "/content/bus_bra_data"
os.makedirs(BREAST_DIR, exist_ok=True) # Create the directory if it doesn't exist

# Download the breast ultrasound dataset from Kaggle
!kaggle datasets download -d orvile/bus-bra-a-breast-ultrasound-dataset -p {BREAST_DIR}

# Unzip all downloaded zip files in the breast ultrasound data directory
for f in glob.glob(os.path.join(BREAST_DIR, "*.zip")):
    with zipfile.ZipFile(f, "r") as z:
        z.extractall(BREAST_DIR)

print("Dataset extracted.")

In [ ]:
# -------------------------------------------------------------------
# -----Apple pretrained CNN & ViT onto Breast Ultrasound Dataset-----
# -------------------------------------------------------------------
import os, glob
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
BREAST_DIR = "/content/bus_bra_data"
IMG_DIR  = os.path.join(BREAST_DIR, "BUSBRA", "BUSBRA", "Images")
MASK_DIR = os.path.join(BREAST_DIR, "BUSBRA", "BUSBRA", "Masks")

# Pair images and masks
image_files = sorted(glob.glob(os.path.join(IMG_DIR, "*.png")))
mask_files  = sorted(glob.glob(os.path.join(MASK_DIR, "*.png")))

mask_lookup = {Path(m).name.replace("mask_", "bus_"): m for m in mask_files}
paired = []
for img_path in image_files:
    name = Path(img_path).name
    if name in mask_lookup:
        paired.append({'image_path': img_path, 'mask_path': mask_lookup[name]})

breast_df = pd.DataFrame(paired)
print(f"Paired: {len(breast_df)} image-mask pairs")

# Transform
TARGET_H, TARGET_W = 384, 576

val_aug = A.Compose([
    A.Resize(TARGET_H, TARGET_W),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Dataset
class BreastUltrasoundDataset(Dataset):
    def __init__(self, rows, transform=None):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image = np.array(Image.open(row['image_path']).convert("RGB"))
        mask = np.array(Image.open(row['mask_path']).convert("L"))
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

breast_ds = BreastUltrasoundDataset(breast_df, transform=val_aug)
breast_loader = DataLoader(breast_ds, batch_size=4, shuffle=False, num_workers=2)

images, masks = next(iter(breast_loader))
print(f"Image batch: {images.shape}, Mask batch: {masks.shape}")

# Load thyroid-trained CNN
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b4",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
    decoder_dropout=0.25,
)
model.load_state_dict(torch.load("best_cnn_model.pth", map_location=device))
model = model.to(device)
model.eval()

# TTA predict
def tta_predict(model, images):
    preds = torch.sigmoid(model(images))
    flipped_h = torch.flip(images, dims=[3])
    preds += torch.flip(torch.sigmoid(model(flipped_h)), dims=[3])
    flipped_v = torch.flip(images, dims=[2])
    preds += torch.flip(torch.sigmoid(model(flipped_v)), dims=[2])
    return preds / 3.0

# Evaluate on breast data
try:
    thresh = best_test_thresh
except NameError:
    thresh = 0.5
    print(f"best_test_thresh not found, using default: {thresh}")

test_iou, test_dice = 0.0, 0.0
with torch.no_grad():
    for images, masks in tqdm(breast_loader, desc="CNN on Breast"):
        images, masks = images.to(device), masks.to(device)
        preds_raw = tta_predict(model, images)
        pred_bin = (preds_raw > thresh).float().squeeze(1)

        smooth = 1e-5
        inter = (pred_bin * masks.float()).sum(dim=(1, 2))
        union = pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) - inter

        test_iou += ((inter + smooth) / (union + smooth)).mean().item()
        test_dice += ((2 * inter + smooth) /
                      (pred_bin.sum(dim=(1, 2)) + masks.float().sum(dim=(1, 2)) + smooth)).mean().item()

n = len(breast_loader)
print(f"\n{'='*50}")
print(f"CNN (UNet++) on BUS-BRA Data (thresh={thresh:.2f}):")
print(f"  Dice: {test_dice / n:.4f}")
print(f"  IoU:  {test_iou / n:.4f}")
print(f"{'='*50}")

# Visualize a few predictions
images, masks = next(iter(breast_loader))
images = images.to(device)

with torch.no_grad():
    preds = tta_predict(model, images)
    preds = (preds > thresh).float().cpu()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(min(3, images.shape[0])):
    img = images[i].cpu()
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + \
          torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    img = img.permute(1, 2, 0).clamp(0, 1)

    axes[i][0].imshow(img)
    axes[i][0].set_title("BUS-BRA Input")
    axes[i][1].imshow(masks[i], cmap="gray")
    axes[i][1].set_title("Ground Truth")
    axes[i][2].imshow(preds[i].squeeze(), cmap="gray")
    axes[i][2].set_title("CNN Prediction")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Cross-Domain: Thyroid-Trained CNN on BUS-BRA Dataset", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Load thyroid-trained SegFormer
vit_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=2,
    ignore_mismatched_sizes=True,
)
vit_model.load_state_dict(torch.load("best_vit_model.pth", map_location=device))
vit_model = vit_model.to(device)
vit_model.eval()
print("SegFormer loaded.")

# Predict helper
def segformer_predict(model, images):
    outputs = model(pixel_values=images)
    logits = outputs.logits
    logits = F.interpolate(logits, size=images.shape[-2:],
                           mode="bilinear", align_corners=False)
    return logits  # [B, 2, H, W]

# Evaluate on breast data
THRESHOLDS = np.arange(0.30, 0.70, 0.05)
test_iou_vit, test_dice_vit = 0.0, 0.0

with torch.no_grad():
    for images, masks in tqdm(breast_loader, desc="SegFormer on Breast"):
        images, masks = images.to(device), masks.to(device)
        logits = segformer_predict(vit_model, images)
        probs = torch.softmax(logits, dim=1)[:, 1, :, :]  # prob of lesion class

        # Best threshold search
        target_f = masks.float()
        smooth = 1e-5
        best_iou_batch, best_dice_batch = 0.0, 0.0
        for t in THRESHOLDS:
            pred = (probs > t).float()
            intersection = (pred * target_f).sum(dim=(1, 2))
            union = pred.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) - intersection
            iou  = ((intersection + smooth) / (union + smooth)).mean().item()
            dice = ((2 * intersection + smooth) /
                    (pred.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) + smooth)).mean().item()
            if dice > best_dice_batch:
                best_iou_batch, best_dice_batch = iou, dice

        test_iou_vit += best_iou_batch
        test_dice_vit += best_dice_batch

n_vit = len(breast_loader)
print(f"\n{'='*50}")
print(f"SegFormer (MiT-B2) on BUS-BRA Data:")
print(f"  Dice: {test_dice_vit / n_vit:.4f}")
print(f"  IoU:  {test_iou_vit / n_vit:.4f}")
print(f"{'='*50}")

# Visualize predictions
images, masks = next(iter(breast_loader))
images = images.to(device)

with torch.no_grad():
    logits = segformer_predict(vit_model, images)
    preds = (torch.softmax(logits, dim=1)[:, 1, :, :] > 0.5).float().cpu()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(min(3, images.shape[0])):
    img = images[i].cpu()
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + \
          torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    img = img.permute(1, 2, 0).clamp(0, 1)

    axes[i][0].imshow(img)
    axes[i][0].set_title("BUS-BRA Input")
    axes[i][1].imshow(masks[i], cmap="gray")
    axes[i][1].set_title("Ground Truth")
    axes[i][2].imshow(preds[i].squeeze(), cmap="gray")
    axes[i][2].set_title("SegFormer Prediction")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Cross-Domain: Thyroid-Trained SegFormer on BUS-BRA Dataset", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Cross-Domain Summary
cnn_dice_avg = test_dice / n
cnn_iou_avg = test_iou / n
vit_dice_avg = test_dice_vit / n_vit
vit_iou_avg = test_iou_vit / n_vit

print(f"\n{'='*50}")
print("Cross-Domain Evaluation Summary (BUS-BRA)")
print(f"{'='*50}")
print(f"  {'Model':<20} {'Dice':>8} {'IoU':>8}")
print(f"  {'-'*36}")
print(f"  {'UNet++ (CNN)':<20} {cnn_dice_avg:>8.4f} {cnn_iou_avg:>8.4f}")
print(f"  {'SegFormer (ViT)':<20} {vit_dice_avg:>8.4f} {vit_iou_avg:>8.4f}")
print(f"{'='*50}")